In [1]:
import pandas as pd
import requests
from zipfile import ZipFile
from io import BytesIO
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
import gzip
import shutil
import os
import json
import numpy as np


### Load data

In [ ]:
df_train = pd.read_csv("./MeetingBank-transcript/train.csv", delimiter=',')
df_train.head()

,meeting_id,source,type,reference,city
0,LongBeachCC_08092022_22-0922,Speaker 4: Thank you. And can we do the functi...,Agenda Item,Recommendation to increase appropriations in t...,LongBeachCC
1,LongBeachCC_08092022_22-0917,Speaker 4: We're going to hear all of the pres...,Public Hearing,Recommendation to conduct a Budget Hearing to ...,LongBeachCC
2,LongBeachCC_08092022_22-0946,Speaker 4: Thank you very much. We will. We're...,Ordinance,Recommendation to declare ordinance amending t...,LongBeachCC
3,LongBeachCC_08092022_22-0932,"Speaker 4: Great. Thank you. And, you know, we...",Resolution,Recommendation to adopt resolution approving t...,LongBeachCC
4,LongBeachCC_08092022_22-0926,Speaker 0: The motion is carried nine zero.\nS...,Agenda Item,Recommendation to refer to the Public Health a...,LongBeachCC


In [ ]:
df_test = pd.read_csv("./MeetingBank-transcript/test.csv", delimiter=',')
df_test.head()
# Count words for all source
df_test['word_count'] = df_test['source'].apply(lambda x: len(x.split()))

# Calculate mean and standard deviation
mean_word_count = df_test['word_count'].mean()
std_word_count = df_test['word_count'].std()

print(f"Mean word count: {mean_word_count}")
print(f"Standard deviation of word count: {std_word_count}")


Mean word count: 2775.7587006960557
Standard deviation of word count: 5610.365060635552


In [ ]:
import pandas as pd
path_csv_test = r".\dataset\test_data.csv"

df_test = pd.read_csv(path_csv_test)# Count words for all source
df_test['word_count'] = df_test['Description'].apply(lambda x: len(x.split()))

# Calculate mean and standard deviation
mean_word_count = df_test['word_count'].mean()
std_word_count = df_test['word_count'].std()

print(f"Mean word count: {mean_word_count}")
print(f"Standard deviation of word count: {std_word_count}")

Mean word count: 17.241655540720963
Standard deviation of word count: 10.897730720930138


In [ ]:
df_val = pd.read_csv("./MeetingBank-transcript/val.csv", delimiter=',')
df_val.head()

,meeting_id,source,type,reference,city
0,LongBeachCC_08092022_22-0949,Speaker 4: Thank you. And then we have our sec...,Contract,"Recommendation to authorize City Manager, or d...",LongBeachCC
1,LongBeachCC_08092022_22-0948,Speaker 0: Item 29 is a report from the City A...,Resolution,Recommendation to adopt resolution of the City...,LongBeachCC
2,LongBeachCC_08092022_22-0758,Speaker 4: Thank you. That concludes that item...,Ordinance,Recommendation to declare ordinance amending t...,LongBeachCC
3,LongBeachCC_08092022_22-0935,Speaker 0: Item 25 is a report from technology...,Contract,Recommendation to adopt Specifications No. ITB...,LongBeachCC
4,LongBeachCC_07192022_22-0828,Speaker 1: One Transfer Item 18 Communication ...,Agenda Item,Recommendation to increase appropriations in t...,LongBeachCC


### Setup LLM

In [ ]:
key = ""

In [ ]:
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "{key}"

llm = ChatOpenAI(model_name="gpt-4o-mini")



### Test

In [6]:
# Test the llm.__call__ method with a sample prompt
test_prompt = "Quel est le résumé de la réunion suivante : Speaker 1: This is a test transcript for the meeting."
test_response = llm.invoke(test_prompt, max_tokens=100)
print(f"Test response: {test_response}")

Test response: content="Il semble que la transcription fournie soit très limitée et ne contient qu'une seule phrase de l'intervenant 1, sans plus de détails sur le contenu de la réunion. Pour un résumé de la réunion, il serait nécessaire d'avoir plus d'informations sur les sujets abordés, les discussions, les décisions prises et les actions à suivre. Si vous avez d'autres parties de la transcription ou des éléments supplémentaires, je serais heureux de vous aider à en faire un résumé." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 29, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_00428b782a', 'finish_reason': 'stop', 'logprobs': None} id='run-a2b708c7-1df5-4095-bb01-200152dcd92

# Prompt

In [9]:
prompt_template = """
You are an assistant specialized in meeting text analysis. Your mission is to extract, from the following transcript, only the task CATEGORIES and their corresponding DESCRIPTIONS that must be completed as a result of the meeting.  

### Categories:  
1. **Document Writing** (e.g., writing reports, drafting bills, composing resolutions)  
2. **Meeting Planning** (e.g., scheduling meetings, organizing sessions, setting dates, preparing interviews)  
3. **Administrative Communication** (e.g., sending official emails, press releases, memos, text messages)  

### Instructions:  
- Extract **only** the tasks explicitly assigned or required for follow-up.  
- Ignore general discussions, suggestions, or ideas that do not translate into concrete actions.  
- Return the result in a structured, **parsable** JSON format.  

### **Expected Output Format:** 
# Leave task empty if no task is found 
```json
{
  "tasks": [
    {
      "category": "Document Writing",
      "description": "Draft the quarterly financial report."
    }, 
    {
      "category": "Meeting Planning",
      "description": "Schedule the next budget review meeting for Friday."
    },
    {
      "category": "Administrative Communication",
      "description": "Send out email reminders for the upcoming stakeholder meeting."
    }
  ]
}```
"""
context = "Here is the meeting transcript: {transcript}"


In [10]:
import json
import re

def parse_tasks(json_data):
    # Initialize the categories dictionary
    categories = {
        "Document Writing": "",
        "Meeting Planning": "",
        "Administrative Communication": ""
    }

    # Ensure input is a string
    if isinstance(json_data, str):
        # Remove Markdown-style code block markers (```json ... ```)
        json_data = re.sub(r"^```json|```$", "", json_data, flags=re.MULTILINE).strip()
        
        try:
            json_data = json.loads(json_data)  # Convert cleaned JSON string to dictionary
        except json.JSONDecodeError:
            print("Error: Invalid JSON format after cleaning")
            return categories  # Return empty categories instead of crashing

    # Iterate through the tasks and append descriptions to the corresponding category
    for task in json_data.get("tasks", []):
        category = task["category"]
        description = task["description"]
        
        if category in categories:
            categories[category] += " | " + description if categories[category] else description

    return categories

# Example usage with unmodified input
json_input = """
```json
{
  "tasks": [
    {
      "category": "Meeting Planning",
      "description": "Prepare for the consideration of Resolutions 348, 349, 350, and 351 on Monday, April 18."
    }
  ]
}"""


parsed_output = parse_tasks(json_input)


In [14]:
import ctypes

# Désactiver la mise en veille
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)  # ES_CONTINUOUS | ES_SYSTEM_REQUIRED

# Ton code ici...

# Réactiver la mise en veille à la fin
ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)  # ES_CONTINUOUS


-2147483646

In [15]:
import logging

# Disable all logging
logging.disable(logging.CRITICAL)


In [16]:
import time
import threading
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import RateLimitError
from tqdm import tqdm

def save_meeting_actions_to_csv(df, filename):
    results = pd.DataFrame(columns=[
        'meeting_id', 'Category_Document_Writing', 'Category_Meeting_Planning', 
        'Category_Administrative_Communication', 'Description_Document_Writing', 
        'Description_Meeting_Planning', 'Description_Administrative_Communication'
    ])

    # Rate limit management
    prompt_counter = 0
    start_time = time.time()
    prompt_counter_lock = threading.Lock()

    def process_row(i):
        """Processes a single row from the DataFrame."""
        nonlocal prompt_counter, start_time

        try:
            prompt = prompt_template + context.format(transcript=df['source'].iloc[i])
        except IndexError:
            print(f"Index {i} is out of bounds for the DataFrame.")
            return pd.DataFrame()
        except Exception as e:
            print(f"Error formatting prompt for row {i}: {e}")
            return pd.DataFrame()

        retries = 5
        while retries > 0:
            try:
                # Enforce rate limit: max 17 requests per minute
                with prompt_counter_lock:
                    if prompt_counter >= 17:
                        elapsed_time = time.time() - start_time
                        if elapsed_time < 60:
                            print(f"Rate limit reached. Sleeping for {60 - elapsed_time:.2f} seconds.")
                            time.sleep(60 - elapsed_time)  # Wait until next window
                        prompt_counter = 0
                        start_time = time.time()

                response = llm.invoke(prompt).content
                parsed_response = parse_tasks(response)

                new_row = pd.DataFrame({
                    'meeting_id': [df['meeting_id'].iloc[i]],
                    'Category_Document_Writing': [parsed_response['Document Writing'] != ""],
                    'Category_Meeting_Planning': [parsed_response['Meeting Planning'] != ""],
                    'Category_Administrative_Communication': [parsed_response['Administrative Communication'] != ""],
                    'Description_Document_Writing': [parsed_response['Document Writing']],
                    'Description_Meeting_Planning': [parsed_response['Meeting Planning']],
                    'Description_Administrative_Communication': [parsed_response['Administrative Communication']]
                })

                with prompt_counter_lock:
                    prompt_counter += 1

                return new_row

            except RateLimitError:
                retries -= 1
                if retries > 0:
                    print(f"Rate limit hit. Retrying in 2.43s... ({retries} retries left)")
                    time.sleep(2.43)
                else:
                    print(f"Max retries reached for row {i}. Skipping.")
                    return pd.DataFrame()

            except Exception as e:
                print(f"Unexpected error processing row {i}: {e}")
                return pd.DataFrame()

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(process_row, i) for i in range(len(df))]

        batch_results = []
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing rows", ncols=100):
            result = future.result()
            if not result.empty:
                batch_results.append(result)

            if len(batch_results) >= 100:
                results = pd.concat([results] + batch_results, ignore_index=True)
                results.to_csv(filename, index=False)
                batch_results = []

        if batch_results:
            results = pd.concat([results] + batch_results, ignore_index=True)
            results.to_csv(filename, index=False)

# Example usage
save_meeting_actions_to_csv(df_val, "meeting_actions_val.csv")
save_meeting_actions_to_csv(df_test, "meeting_actions_test.csv")
save_meeting_actions_to_csv(df_train, "meeting_actions_train.csv")

Processing rows:   2%|▉                                            | 17/861 [00:10<12:35,  1.12it/s]

Rate limit reached. Sleeping for 49.18 seconds.


Processing rows:   4%|█▊                                           | 34/861 [01:02<16:13,  1.18s/it]

Rate limit reached. Sleeping for 56.75 seconds.


Processing rows:   6%|██▋                                          | 51/861 [02:05<21:11,  1.57s/it]

...

Processing rows:  99%|███████████████████████████████████████▋| 5133/5169 [5:01:05<01:00,  1.67s/it]

Rate limit reached. Sleeping for 57.29 seconds.


Processing rows: 100%|███████████████████████████████████████▊| 5150/5169 [5:02:04<00:28,  1.52s/it]

Rate limit reached. Sleeping for 57.74 seconds.


Processing rows: 100%|████████████████████████████████████████| 5169/5169 [5:03:07<00:00,  3.52s/it]


In [17]:
# Réactiver la mise en veille à la fin
ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)  # ES_CONTINUOUS

-2147483648